In [1]:
import logging

In [ ]:
request_logger = logging.getLogger("../logs/A_log")
request_logger.setLevel(logging.INFO)

request_handler = logging.FileHandler("../logs/A_log.log", mode='w')
request_formatter = logging.Formatter("%(name)s %(asctime)s %(levelname)s %(message)s")

request_handler.setFormatter(request_formatter)
request_logger.addHandler(request_handler)

request_logger.info(f"Логгирование сборки датасета A_log...")

In [3]:
import pandas as pd
import time

In [4]:
GROUP_IDS = {
    '-220754053': ['VK Видео'                             , 'vkvideo'           ],
    '-81597813' : ['Влад Бумага А4'                       , 'a4'                ],
    '-211169870': ['ГЛЕНТ'                                , 'glent'             ],
    '-152009330': ['ИКС'                                  , 'iks__club'         ],
    '-219283548': ['Вильям Бруно'                         , 'brunobroo'         ],
    '-147169109': ['GEO'                                  , 'geeeo'             ],
    '-213802301': ['ШАСТУН'                               , 'shastoon.channel'  ],
    '-195985818': ['Hardcore Fighting Championship'       , 'hardcore.fighting' ],
    '-218565915': ['Асафьев Стас'                         , 'asafevstas'        ],
    '-219482354': ['Варвара Щербакова по вашим интересам!', 'varjauletela_group'],
    '-221130436': ['Парковка'                             , 'parkovka_show'     ],
    '-211022028': ['Арай Чобанян'                         , 'arai_papa'         ],
    '-218471730': ['Комьюнити'                            , 'community_show'    ],
    '-211220744': ['Стрелец-Молодец'                      , 'strelets_molodec'  ],
    '-1415705'  : ['GAZ'                                  , 'gaz'               ],
    '-109800058': ['Medium Quality Channel'               , 'mediumquality'     ],
}

In [19]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('API_A.ipynb'), '..')))

from py.API_methods import get_all_videos, get_unique_album_videos, get_group_info, get_wall_info, get_video_comment

### Выгружаем основной датасет с видео

In [6]:
all_videos = []

for group, group_name in GROUP_IDS.items():
    request_logger.info(f"Достаем информацию из сообщества: {group_name[0]}...")

    request_logger.info(f"Получаем все видео из сообщества: {group_name[0]}...")
    videos = get_all_videos(group)

    total_videos = len(videos)
    request_logger.info(f"Всего видео: {total_videos}")

    time.sleep(1)
    
    request_logger.info(f"Получаем количество видео в альбомах из сообщества: {group_name[0]}...")
    album_videos = get_unique_album_videos(group)

    album_video_percent = round((album_videos / total_videos) * 100, 2) if total_videos > 0 else 0
    request_logger.info(f"Уникальных видео в альбомах: {album_videos} ({album_video_percent}%)")

    for video in videos:
        video["group_name"] = group_name[0]
        video["album_video_percent"] = album_video_percent
        all_videos.append(video)

    time.sleep(1) 

df = pd.DataFrame(all_videos)

cols_to_keep = [
    "id"        , "owner_id"   , "group_name" , "title"        , "description", "duration", "date",
    "views"     , "comments"   , "likes_count", "reposts_count", "player"     , "can_like",
    "can_repost", "can_dislike", "is_pinned"  , "image_url"    , "album_video_percent"    ,
]
existing_cols = [col for col in cols_to_keep if col in df.columns]
df = df[existing_cols]

In [7]:
df.shape

(8010, 15)

In [8]:
request_logger.info("Сохрнаяем итоговую таблицу в файл")
csv_filename = "csv_files/videos_1.csv"
df.to_csv(csv_filename, index=False, encoding="utf-8")

request_logger.info(f"Итоговый файл сохранён: {csv_filename}")

### Выгружаем доп инфу о сообещствах

In [9]:
group_ids = ",".join([value[1] for value in GROUP_IDS.values()])

optional_fields = ",".join([
    "description", "members_count", "activity", "status"    , "contacts", "ban_info",
    "links"      , "verified"     , "site"    , "age_limits", "banned"  , "city"    ,
    "country"    , "place"        ,
])

request_logger.info("Выгружаем доп информацию о сообещствах")
groups_data = get_group_info(group_ids, optional_fields)

if groups_data:
    request_logger.info("Успешно собрали")

    group_info = pd.DataFrame(groups_data.get("groups"))
    request_logger.info("Сохрнаяем итоговую таблицу в файл")

    csv_filename = "csv_files/groups_1.csv"
    group_info.to_csv(csv_filename, index=False, encoding="utf-8")

    request_logger.info(f"Итоговый файл сохранён: {csv_filename}")
else:
    request_logger.warning("Информация о сообществах не собралась")

In [10]:
group_info.shape

(16, 20)

### Получаем посты сообществ

In [11]:
all_posts = []

for id, items in GROUP_IDS.items():
    request_logger.info(f"Получение постов для сообщества: {items[0]}")
    wall_posts_data = get_wall_info(id, items[1])

    if wall_posts_data:
        request_logger.info("Успешно собрали")
        all_posts.extend(wall_posts_data)
    else:
        request_logger.warning("Информация о постах не собралась")

    time.sleep(0.5)

if all_posts != []:
    request_logger.info("Успешно собрали все посты")

    posts = pd.DataFrame(all_posts)
    request_logger.info("Сохрнаяем итоговую таблицу в файл")

    csv_filename = "csv_files/posts_1.csv"
    posts.to_csv(csv_filename, index=False, encoding="utf-8")

    request_logger.info(f"Итоговый файл сохранён: {csv_filename}")
else:
    request_logger.warning("Информация о постах не собралась")

In [12]:
posts.shape

(1616, 31)

### Получаем последний комментарий к видео

In [13]:
df_coments = pd.read_csv("csv_files/videos_1.csv", usecols=['id', 'owner_id', 'title'])

In [14]:
df['top_comment_text'] = ""  

batch_size = 10
video_groups = [df.iloc[i:i + batch_size] for i in range(0, len(df), batch_size)]

for batch in video_groups:
    get_video_comment(batch, df)


# request_logger.info("Сохрнаяем итоговую таблицу в файл")
# csv_filename = "csv_files/comments_1.csv"
# df.to_csv(csv_filename, index=False, encoding="utf-8")

# request_logger.info(f"Итоговый файл сохранён: {csv_filename}")